In [1]:
import pandas as pd
from datasets import load_dataset


In [2]:
data=load_dataset("hotpotqa/hotpot_qa","distractor")

In [3]:
df=data['train'].to_pandas()
df_val=data['validation'].to_pandas()

In [5]:
print(df['context'][1])
print()
print(df['supporting_facts'][1])
print(df.shape)
df["type"].value_counts()

{'title': array(['Ritz-Carlton Jakarta', 'Oberoi family', 'Ishqbaaaz',
       'Hotel Tallcorn', 'Mohan Singh Oberoi', 'Hotel Bond',
       'The Oberoi Group', 'Future Fibre Technologies',
       '289th Military Police Company', 'Glennwanis Hotel'], dtype=object), 'sentences': array([array(['The Ritz-Carlton Jakarta is a hotel and skyscraper in Jakarta, Indonesia and 14th Tallest building in Jakarta.',
              ' It is located in city center of Jakarta, near Mega Kuningan, adjacent to the sister JW Marriott Hotel.',
              ' It is operated by The Ritz-Carlton Hotel Company.',
              ' The complex has two towers that comprises a hotel and the Airlangga Apartment respectively.',
              ' The hotel was opened in 2005.'], dtype=object)                                                                 ,
       array(['The Oberoi family is an Indian family that is famous for its involvement in hotels, namely through The Oberoi Group.'],
             dtype=object)      

type
bridge        72991
comparison    17456
Name: count, dtype: int64

In [4]:
df["num_gold_paragraphs"]=df["supporting_facts"].apply(lambda x : len(set(x['title'])))
print(df["num_gold_paragraphs"].value_counts())
print(df.groupby("type")["num_gold_paragraphs"].mean())
df_val["num_gold_paragraphs"]=df_val["supporting_facts"].apply(lambda x : len(set(x['title'])))
print(df_val["num_gold_paragraphs"].value_counts())
print(df_val.groupby("type")["num_gold_paragraphs"].mean())

num_gold_paragraphs
2    90447
Name: count, dtype: int64
type
bridge        2.0
comparison    2.0
Name: num_gold_paragraphs, dtype: float64
num_gold_paragraphs
2    7405
Name: count, dtype: int64
type
bridge        2.0
comparison    2.0
Name: num_gold_paragraphs, dtype: float64


In [5]:
def titles_in_q(df):
    q=df["question"].lower()
    titles=set(df['supporting_facts']['title'])
    return sum(1 for t in titles if t.lower() in q)

df["num_titles_in_question"]=df.apply(titles_in_q, axis=1)
print(df.groupby("type")["num_titles_in_question"].value_counts())
df_val["num_titles_in_question"]=df_val.apply(titles_in_q, axis=1)
print(df_val.groupby("type")["num_titles_in_question"].value_counts())


type        num_titles_in_question
bridge      1                         37788
            0                         30510
            2                          4693
comparison  2                         13455
            1                          3470
            0                           531
Name: count, dtype: int64
type        num_titles_in_question
bridge      1                         3098
            0                         2521
            2                          299
comparison  2                         1152
            1                          287
            0                           48
Name: count, dtype: int64


In [10]:
for col in df.columns:
    print(col,df[col].isnull().sum())
print("Empty_rows in 'question':", (df["question"].str.strip()=="").sum())
print("Rows with empty context: ",df["context"].apply(lambda x : len(x['title'])==0).sum())
print("\nFor Validation Columns\n")
for col in df_val.columns : 
    print(col,df_val[col].isnull().sum())
print("Empty_rows in 'question':", (df_val["question"].str.strip()=="").sum())
print("Rows with empty context: ",df_val["context"].apply(lambda x : len(x['title'])==0).sum())


id 0
question 0
answer 0
type 0
level 0
supporting_facts 0
context 0
num_gold_paragraphs 0
num_titles_in_question 0
number_sf_in_context 0
Empty_rows in 'question': 0
Rows with empty context:  0

For Validation Columns

id 0
question 0
answer 0
type 0
level 0
supporting_facts 0
context 0
num_gold_paragraphs 0
num_titles_in_question 0
Empty_rows in 'question': 0
Rows with empty context:  0


In [13]:
print("Duplicate_questions in train: ",df.duplicated(subset=["question"]).sum())
print("Duplicate_questions in validation: ",df_val.duplicated(subset=["question"]).sum())
def fact_match(row):
    context=set(row['context']['title'])
    sf=set(row['supporting_facts']['title'])
    return sum(1 for t in sf if t in context)

df["number_sf_in_context"]=df.apply(fact_match,axis=1)
df_val["number_sf_in_context"]=df_val.apply(fact_match,axis=1)
print((df["number_sf_in_context"]==df["supporting_facts"].apply(lambda x : len(set(x['title'])))).sum())
print((df_val['number_sf_in_context']==df_val["supporting_facts"].apply(lambda x : len(set(x['title'])))).sum())

Duplicate_questions in train:  7
Duplicate_questions in validation:  0
90447
7405


In [15]:
df_val.to_parquet("../data/processed/df_val_enriched.parquet")